In [ ]:
from pathlib import Path
import json
import re
import importlib

import pandas as pd
from tqdm import tqdm

try:
    psycopg2 = importlib.import_module("psycopg2")
    execute_values = importlib.import_module("psycopg2.extras").execute_values
except ImportError:
    psycopg2 = None
    execute_values = None

# ===================== CONFIG =====================
ARCHIVES = [109]
RESULT_ROOT = Path("../result")
OUTPUT_CSV = Path("./result_101/experiment_results_powerbi_archive109_json.csv")
INCLUDE_TLE = True

IMPORT_TO_DB = False
DB_CONFIG_PATH = Path("./postgres_config.json")
DB_SCHEMA = "subopt"
DB_TABLE = "experiment_run"
# ==================================================


def discover_json_files(archives, result_root: Path):
    files = []
    for a in archives:
        root = result_root / f"archive-{a}"
        if not root.exists():
            print(f"[WARN] Missing archive path: {root}")
            continue
        files.extend(root.rglob("*.json"))
    return sorted(files)


def safe_float(v):
    try:
        return float(v)
    except Exception:
        return None


def safe_int(v):
    try:
        return int(v)
    except Exception:
        return None


def parse_legacy_from_path_and_name(p: Path):
    task = None
    ground_size = None
    seed = None
    try:
        seed = safe_int(p.parent.parent.name)
        ground_size = safe_int(p.parent.name)
        task = p.parent.parent.parent.name
    except Exception:
        pass

    parts = p.stem.split("-")
    algo = strategy = ub = d_mode = budget = alpha = model = None
    if len(parts) >= 6:
        model = parts[-1]
        alpha = safe_float(parts[-2])
        budget = safe_float(parts[-3])
        if len(parts) >= 7:
            algo = parts[0]
            strategy = parts[1] if parts[1] != "none" else None
            ub = parts[2]
            d_mode = parts[3]
        else:
            algo = parts[0]
            strategy = None
            ub = parts[1]
            d_mode = parts[2]
    return {
        "Task": task,
        "Ground_Size": ground_size,
        "Seed": seed,
        "Algorithm": algo,
        "Strategy": strategy,
        "UB": ub,
        "D": d_mode,
        "Budget": budget,
        "Alpha": alpha,
        "Model": model,
    }


def build_row_from_json(p: Path):
    with p.open("r", encoding="utf-8") as f:
        obj = json.load(f)

    # Normalized schema written by archive-109 converter:
    # {"meta": {...}, "result": {...}}
    if isinstance(obj, dict) and "result" in obj:
        res = obj.get("result") or {}
        meta = obj.get("meta") or {}
    else:
        res = obj if isinstance(obj, dict) else {}
        meta = res.get("meta") if isinstance(res.get("meta"), dict) else {}

    legacy = parse_legacy_from_path_and_name(p)

    row = {
        "Source_File": str(p),
        "Archive": None,
        "Task": legacy.get("Task"),
        "Ground_Size": legacy.get("Ground_Size"),
        "Seed": legacy.get("Seed"),
        "Algorithm": legacy.get("Algorithm"),
        "Strategy": legacy.get("Strategy"),
        "UB": legacy.get("UB"),
        "D": legacy.get("D"),
        "Budget": legacy.get("Budget"),
        "Alpha": legacy.get("Alpha"),
        "Model": legacy.get("Model"),
        "Objective_f(S)": res.get("f(S)"),
        "Cost_c(S)": res.get("c(S)"),
        "Time_s": res.get("time"),
        "Node_Count": res.get("node_count"),
        "Open_List_Count": res.get("open_list_count"),
        "TLE": res.get("TLE"),
        "Solution_Set_Size": len(res.get("S", [])) if isinstance(res.get("S", []), list) else None,
        "Meta_Version": None,
        "Meta_Created_UTC": None,
        "Meta_Config_Path": None,
        "Meta_Strategy_Raw": None,
        "Flag_LocalSearch": None,
        "Flag_Cascade": None,
        "Flag_NoInherit": None,
        "Flag_Adaptive": None,
        "Adaptive_Ratio": None,
    }

    if isinstance(meta, dict):
        row["Archive"] = meta.get("archive")
        row["Task"] = meta.get("task", row["Task"])
        row["Ground_Size"] = meta.get("ground_size", row["Ground_Size"])
        row["Seed"] = meta.get("seed", row["Seed"])
        row["Algorithm"] = meta.get("algorithm", row["Algorithm"])
        row["Strategy"] = meta.get("strategy", row["Strategy"])
        row["UB"] = meta.get("heuristic", row["UB"])
        row["D"] = meta.get("d", row["D"])
        row["Budget"] = meta.get("budget", row["Budget"])
        row["Alpha"] = meta.get("alpha", row["Alpha"])
        row["Model"] = meta.get("model_class", row["Model"])
        row["Meta_Version"] = meta.get("version")
        row["Meta_Created_UTC"] = meta.get("created_at_utc")
        row["Meta_Config_Path"] = meta.get("config_path")
        row["Meta_Strategy_Raw"] = meta.get("strategy_raw")
        flags = meta.get("flags") or {}
        row["Flag_LocalSearch"] = bool(flags.get("local_search", False))
        row["Flag_Cascade"] = bool(flags.get("cascade", False))
        row["Flag_NoInherit"] = bool(flags.get("no_inherit", False))
        row["Flag_Adaptive"] = bool(flags.get("adaptive", False))
        row["Adaptive_Ratio"] = meta.get("adaptive_ratio")
    else:
        m = re.search(r"archive-(\d+)", str(p).replace("\\", "/"))
        if m:
            row["Archive"] = m.group(1)

    return row


def export_powerbi_csv_from_json(archives, result_root: Path, output_csv: Path, include_tle=True):
    files = discover_json_files(archives, result_root)
    print(f"[INFO] Found {len(files)} json files across archives={archives}")

    rows = []
    errors = []
    for p in tqdm(files, desc="Parsing json"):
        # Skip config/log json files by requiring expected path depth + filename pattern
        if "archive-" not in str(p).replace("\\", "/"):
            continue
        if "-" not in p.stem:
            continue
        try:
            row = build_row_from_json(p)
            if (not include_tle) and bool(row.get("TLE", False)):
                continue
            rows.append(row)
        except Exception as e:
            errors.append({"file": str(p), "error": str(e)})

    df = pd.DataFrame(rows)
    if not df.empty:
        for c in [
            "Archive", "Ground_Size", "Seed", "Budget", "Alpha", "Time_s",
            "Node_Count", "Open_List_Count", "Objective_f(S)", "Cost_c(S)"
        ]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")

        df.sort_values(
            by=["Archive", "Task", "Algorithm", "UB", "Budget", "Seed"],
            inplace=True,
            na_position="last",
        )

    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print(f"[OK] Exported {len(df)} rows -> {output_csv}")
    if errors:
        err_csv = output_csv.with_name(output_csv.stem + "_errors.csv")
        pd.DataFrame(errors).to_csv(err_csv, index=False, encoding="utf-8-sig")
        print(f"[WARN] {len(errors)} parse errors -> {err_csv}")

    return df, errors


def load_db_config(path: Path):
    with path.open("r", encoding="utf-8-sig") as f:
        return json.load(f)


def to_db_records(df: pd.DataFrame):
    if df.empty:
        return []

    tmp = df.copy()
    if "Meta_Created_UTC" in tmp.columns:
        tmp["Meta_Created_UTC"] = pd.to_datetime(tmp["Meta_Created_UTC"], utc=True, errors="coerce")

    def sql_value(v):
        if pd.isna(v):
            return None
        return v

    records = []
    for _, row in tmp.iterrows():
        records.append(
            (
                sql_value(row.get("Source_File")),
                sql_value(row.get("Archive")),
                sql_value(row.get("Task")),
                sql_value(row.get("Ground_Size")),
                sql_value(row.get("Seed")),
                sql_value(row.get("Algorithm")),
                sql_value(row.get("Strategy")),
                sql_value(row.get("Meta_Strategy_Raw")),
                sql_value(row.get("UB")),
                sql_value(row.get("D")),
                sql_value(row.get("Budget")),
                sql_value(row.get("Alpha")),
                sql_value(row.get("Model")),
                sql_value(row.get("Objective_f(S)")),
                sql_value(row.get("Cost_c(S)")),
                sql_value(row.get("Time_s")),
                sql_value(row.get("Node_Count")),
                sql_value(row.get("Open_List_Count")),
                sql_value(row.get("TLE")),
                sql_value(row.get("Solution_Set_Size")),
                sql_value(row.get("Flag_LocalSearch")),
                sql_value(row.get("Flag_Cascade")),
                sql_value(row.get("Flag_NoInherit")),
                sql_value(row.get("Flag_Adaptive")),
                sql_value(row.get("Adaptive_Ratio")),
                sql_value(row.get("Meta_Version")),
                sql_value(row.get("Meta_Created_UTC")),
                sql_value(row.get("Meta_Config_Path")),
            )
        )
    return records


def import_experiment_runs_to_db(df: pd.DataFrame, db_cfg: dict, schema="subopt", table="experiment_run"):
    if psycopg2 is None or execute_values is None:
        raise RuntimeError("psycopg2 is not installed. Install it to enable database import.")

    records = to_db_records(df)
    if not records:
        print("[INFO] No rows to import.")
        return 0

    cols = [
        "source_file", "archive", "task", "ground_size", "seed", "algorithm", "strategy",
        "strategy_raw", "heuristic", "d_mode", "budget", "alpha", "model_class",
        "objective_f_s", "cost_c_s", "time_s", "node_count", "open_list_count", "tle",
        "solution_set_size", "flag_local_search", "flag_cascade", "flag_no_inherit",
        "flag_adaptive", "adaptive_ratio", "meta_version", "meta_created_utc", "meta_config_path",
    ]
    col_csv = ", ".join(cols)
    updates = ", ".join([f"{c}=EXCLUDED.{c}" for c in cols if c != "source_file"])
    sql = f"""
        INSERT INTO {schema}.{table} ({col_csv})
        VALUES %s
        ON CONFLICT (source_file)
        DO UPDATE SET {updates}
    """

    conn = psycopg2.connect(
        host=db_cfg["host"],
        port=db_cfg["port"],
        dbname=db_cfg["database"],
        user=db_cfg["user"],
        password=db_cfg["password"],
    )
    try:
        with conn:
            with conn.cursor() as cur:
                execute_values(cur, sql, records, page_size=500)
        print(f"[OK] Upserted {len(records)} rows into {schema}.{table}")
    finally:
        conn.close()
    return len(records)


df_results, parse_errors = export_powerbi_csv_from_json(
    ARCHIVES,
    RESULT_ROOT,
    OUTPUT_CSV,
    include_tle=INCLUDE_TLE,
)

print(df_results.head(10))
print("rows:", len(df_results), "errors:", len(parse_errors))

if IMPORT_TO_DB:
    cfg = load_db_config(DB_CONFIG_PATH)
    inserted = import_experiment_runs_to_db(df_results, cfg, schema=DB_SCHEMA, table=DB_TABLE)
    print("db_upserted_rows:", inserted)
